In [49]:
import os
import certifi
import requests  
from dotenv import load_dotenv

In [ ]:
from langchain.chat_models import init_chat_model
# from langchain_google_genai import ChatGoogleGenerativeAI 

# this is for internet search operation
from langchain_tavily import TavilySearch 
from langchain.tools import tool
im

In [ ]:
from langchain.agents import create_agent

In [ ]:
# Load Environment Variable
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")


In [ ]:
search_tool = TavilySearch(max_results=2)

In [ ]:
@tool 
def get_weather(city:str)->str:
    """Fetch the current weather of the city using webstack api"""
    api_key= os.getenv("WEATHERSTACK_API_KEY")
    if not api_key:
        raise ValueError("Api key is not available")
    
    # Its a free tier so we are using HTTP
    url = "https://openweathermap.org"
    params = {"access_key": api_key, "query": city}
    try:
        response = requests.get(url,params=params, timeout=10)
        data = response.json()

        if "error" in data:
            return  f"Weatherstack API error: {data['error'].get('info', 'Unknown error')}"
        
        location = data["location"]["name"]
        country = data["location"]["country"]
        temp = data["current"]["temperature"]
        feels_like = data["current"]["feelslike"]
        humidity = data["current"]["humidity"]
        desc = ", ".join(data["current"]["weather_descriptions"])

        return (
            f"Current weather in {location}, {country}: {desc}. "
            f"Temperature: {temp}°C (feels like {feels_like}°C), Humidity: {humidity}%."
        )

    except Exception as e:
        return f"Failed to fetch weather data: {str(e)}"


In [ ]:
# Testig the Search Tool 
response = search_tool.invoke("What is the capital of jharkahnd")
for item in response["results"]:
    print(item["content"])

In [34]:
#Initialising LLM

llm = init_chat_model(
    "gemini-3.1-flash-lite",
    model_provider="google_genai",
    thinking_budget=0,
)

In [46]:
agent=create_agent(
    model=llm,
    tools=[search_tool,get_weather],
    system_prompt="You are a helpful AI assistant. Use the search tool to find accurate, up-to-date information."
)


In [39]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What year is this?"}]
})
response

{'messages': [HumanMessage(content='What year is this?', additional_kwargs={}, response_metadata={}, id='2004abf2-8e80-4325-ad0a-60def40e09ee'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current year"}'}, '__gemini_function_call_thought_signatures__': {'call_2849987800391783443': 'EqEBCp4BAWkUfROGxCWasdyte5dOdkSM74kinyKoZ6ABB4yiDalRQJSS+I8Bc6rlLvIaNo5vL2snOu1Vbzg2dMDYyEy3ZLsmAursyoqLxwX/gWJdmaXHo+0jK4+oShizgoXPEf9C32URmg77WYjVQ9ff+nHojRceay7DXw6IGzC/UarXa7AoayeFhCGnrSIGDmblCdo7Q9VuBuiMe66FGGeNTyQ='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'}, id='lc_run--01a20354-6231-739b-8333-d397d734a593-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current year'}, 'id': 'call_2849987800391783443', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1518,

In [48]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather in india?"}]
})
response

{'messages': [HumanMessage(content='What is the weather in india?', additional_kwargs={}, response_metadata={}, id='a0a2fe83-d120-48af-8ada-7932ad8f49fa'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current weather in India"}'}, '__gemini_function_call_thought_signatures__': {'call_6469063172308559541': 'EqEBCp4BAWkUfRPyb2ofFqF1wdsfFE5TMGiLHv//ILzI31orreTAqLHvDOmXVsKUpMfU3Wc422I5K5FNIhfo+26+t8A9n+aYs6rYT4Qd2ukfKR2wQrdTLaLCw6mPiOn8OJAiDbQFc1Td9RFDT3XJFuA+8Jb2ezvTeGXr9Q6Mze7L1kV+aDeBI+Y/87LtH5g1fUVcZj4JP8w6tSVjWeRBs5h/J1w='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'tool_call', 'model_provider': 'google_genai'}, id='lc_run--01a2044d-613c-71b8-87e9-39254c5cc9ee-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current weather in India'}, 'id': 'call_6469063172308559541', 'type': 'tool_call'}], invalid_tool_calls=[], us